In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path.cwd().parent
SYNTHETIC_DIR = BASE_DIR / "data" / "synthetic"
CONTROL_DIR = SYNTHETIC_DIR / "control_outputs"

CONTROL_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("02.33 - FIX ANOMALY TRUTH OUTPUT")
print("=" * 70)

# ------------------------------------------------------------
# Scenario folders that contain controlled anomalies/events
# ------------------------------------------------------------

SCENARIOS = {
    "missing_value_adjusted": "missing_value",
    "duplicate_row_adjusted": "duplicate_row",
    "invalid_data_adjusted": "invalid_data",
    "outlier_adjusted": "outlier",
    "inconsistent_text_category_adjusted": "inconsistent_text_category",
    "hidden_business_events_adjusted": "hidden_business_event",
    "pricing_discount_adjusted": "pricing_discount",
    "cancellation_adjusted": "cancellation",
    "channel_payment_adjusted": "channel_payment",
    "seasonality_adjusted": "seasonality",
    "weekday_weekend_adjusted": "weekday_weekend",
    "target_growth_adjusted": "target_growth",
    "customer_behavior_adjusted": "customer_behavior",
    "product_category_adjusted": "product_category",
    "store_regional_adjusted": "store_regional",
    "schema_change_adjusted": "schema_change",
    "exact_duplicate_file_adjusted": "exact_duplicate_file",
    "correction_file_adjusted": "correction_file",
    "future_incremental_batches": "future_incremental"
}

# ------------------------------------------------------------
# Business-key columns
# ------------------------------------------------------------

KEY_COLUMNS = [
    "sales_line_id",
    "return_id",
    "target_id",
    "customer_id",
    "product_id",
    "store_id",
    "salesperson_id",
    "promotion_id",
    "order_id"
]

truth_records = []

# ------------------------------------------------------------
# Find anomaly/event files
# ------------------------------------------------------------

for folder_name, anomaly_type in SCENARIOS.items():

    scenario_dir = SYNTHETIC_DIR / folder_name

    if not scenario_dir.exists():
        continue

    csv_files = list(scenario_dir.rglob("*.csv"))

    print(
        f"\nScenario: {anomaly_type}"
    )
    print(
        f"Files found: {len(csv_files)}"
    )

    for file_path in csv_files:

        try:
            df = pd.read_csv(file_path)

        except Exception:
            continue

        if df.empty:
            continue

        # ----------------------------------------------------
        # Find the business key available in this file
        # ----------------------------------------------------

        key_col = None

        for candidate in KEY_COLUMNS:
            if candidate in df.columns:
                key_col = candidate
                break

        if key_col is None:
            continue

        # ----------------------------------------------------
        # Detect duplicated business keys
        # ----------------------------------------------------

        duplicate_mask = df[key_col].duplicated(
            keep=False
        )

        duplicate_rows = df.loc[
            duplicate_mask,
            [key_col]
        ]

        for key_value in duplicate_rows[key_col].dropna().astype(str):

            truth_records.append({
                "dataset": file_path.name,
                "business_key": key_value,
                "anomaly_type": anomaly_type,
                "expected_classification": "REVIEW",
                "source_path": str(
                    file_path.relative_to(SYNTHETIC_DIR)
                )
            })

        # ----------------------------------------------------
        # For non-duplicate scenarios, record rows present
        # in the adjusted scenario output.
        # ----------------------------------------------------

        if not duplicate_mask.any():

            for key_value in df[key_col].dropna().astype(str):

                truth_records.append({
                    "dataset": file_path.name,
                    "business_key": key_value,
                    "anomaly_type": anomaly_type,
                    "expected_classification": "REVIEW",
                    "source_path": str(
                        file_path.relative_to(SYNTHETIC_DIR)
                    )
                })


# ------------------------------------------------------------
# Remove exact duplicate truth records
# ------------------------------------------------------------

if truth_records:

    anomaly_truth = pd.DataFrame(
        truth_records
    )

    anomaly_truth = anomaly_truth.drop_duplicates(
        subset=[
            "dataset",
            "business_key",
            "anomaly_type",
            "source_path"
        ]
    ).reset_index(drop=True)

else:

    anomaly_truth = pd.DataFrame(
        columns=[
            "dataset",
            "business_key",
            "anomaly_type",
            "expected_classification",
            "source_path"
        ]
    )


# ------------------------------------------------------------
# Save anomaly truth
# ------------------------------------------------------------

truth_file = CONTROL_DIR / "anomaly_truth.csv"

anomaly_truth.to_csv(
    truth_file,
    index=False
)

print("\n" + "=" * 70)
print("ANOMALY TRUTH RESULT")
print("=" * 70)

print(
    f"Anomaly truth rows : {len(anomaly_truth):,}"
)

print(
    f"Columns            : {len(anomaly_truth.columns)}"
)

print(
    f"Saved to           : {truth_file}"
)

if len(anomaly_truth) > 0:

    print("\nAnomaly types found:")

    print(
        anomaly_truth[
            "anomaly_type"
        ]
        .value_counts()
        .to_string()
    )

    print(
        "\n02.33 anomaly truth fix: PASSED"
    )

else:

    print(
        "\n02.33 anomaly truth fix: NEEDS REVIEW"
    )

02.33 - FIX ANOMALY TRUTH OUTPUT

Scenario: missing_value
Files found: 32

Scenario: duplicate_row
Files found: 32

Scenario: invalid_data
Files found: 32

Scenario: outlier
Files found: 32

Scenario: inconsistent_text_category
Files found: 34

Scenario: hidden_business_event
Files found: 33

Scenario: pricing_discount
Files found: 33

Scenario: cancellation
Files found: 32


KeyboardInterrupt: 

In [7]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
SYNTHETIC_DIR = PROJECT_ROOT / "data" / "synthetic"
CONTROL_DIR = SYNTHETIC_DIR / "control_outputs"

ANOMALY_TRUTH_FILE = CONTROL_DIR / "anomaly_truth.csv"

# Approved anomaly scenario folders
ANOMALY_SCENARIOS = {
    "missing_value": SYNTHETIC_DIR / "missing_value_adjusted",
    "duplicate_row": SYNTHETIC_DIR / "duplicate_row_adjusted",
    "invalid_data": SYNTHETIC_DIR / "invalid_data_adjusted",
    "outlier": SYNTHETIC_DIR / "outlier_adjusted",
}

records = []

print("=" * 70)
print("02.33 - FIX ANOMALY TRUTH OUTPUT")
print("=" * 70)

for anomaly_type, scenario_dir in ANOMALY_SCENARIOS.items():

    files = sorted(scenario_dir.glob("*.csv"))

    print(f"\nScenario: {anomaly_type}")
    print(f"Files found: {len(files)}")

    for file_path in files:

        records.append({
            "business_key": "",
            "anomaly_type": anomaly_type,
            "expected_classification": "REVIEW",
            "source_path": str(file_path.relative_to(SYNTHETIC_DIR))
        })

anomaly_truth = pd.DataFrame(
    records,
    columns=[
        "business_key",
        "anomaly_type",
        "expected_classification",
        "source_path"
    ]
)

anomaly_truth.to_csv(
    ANOMALY_TRUTH_FILE,
    index=False
)

print("\n" + "=" * 70)
print("ANOMALY TRUTH CREATED")
print("=" * 70)

print(f"Rows created : {len(anomaly_truth):,}")
print(f"Columns      : {len(anomaly_truth.columns)}")
print(f"Saved to     : {ANOMALY_TRUTH_FILE}")

print("\nAnomaly distribution:")
print(anomaly_truth["anomaly_type"].value_counts())

print("\n02.33 anomaly_truth fix completed.")


02.33 - FIX ANOMALY TRUTH OUTPUT

Scenario: missing_value
Files found: 0

Scenario: duplicate_row
Files found: 0

Scenario: invalid_data
Files found: 0

Scenario: outlier
Files found: 0

ANOMALY TRUTH CREATED
Rows created : 0
Columns      : 4
Saved to     : c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\control_outputs\anomaly_truth.csv

Anomaly distribution:
Series([], Name: count, dtype: int64)

02.33 anomaly_truth fix completed.


In [8]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SYNTHETIC_DIR = PROJECT_ROOT / "data" / "synthetic"

ANOMALY_SCENARIOS = {
    "missing_value": SYNTHETIC_DIR / "missing_value_adjusted",
    "duplicate_row": SYNTHETIC_DIR / "duplicate_row_adjusted",
    "invalid_data": SYNTHETIC_DIR / "invalid_data_adjusted",
    "outlier": SYNTHETIC_DIR / "outlier_adjusted",
}

print("=" * 70)
print("02.33 - LOCATE ANOMALY OUTPUT FILES")
print("=" * 70)

for anomaly_type, scenario_dir in ANOMALY_SCENARIOS.items():

    print(f"\n{anomaly_type}")
    print(f"Folder: {scenario_dir}")

    if not scenario_dir.exists():
        print("STATUS: FOLDER NOT FOUND")
        continue

    # Recursive search
    files = sorted(scenario_dir.rglob("*.csv"))

    print(f"CSV files found: {len(files)}")

    for file_path in files[:5]:
        print("  ", file_path.relative_to(SYNTHETIC_DIR))

    if len(files) > 5:
        print(f"  ... and {len(files) - 5} more")

print("\n02.33 file-location check completed.")

02.33 - LOCATE ANOMALY OUTPUT FILES

missing_value
Folder: c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\missing_value_adjusted
CSV files found: 32
   missing_value_adjusted\sales_adjusted\sales_2024-01.csv
   missing_value_adjusted\sales_adjusted\sales_2024-02.csv
   missing_value_adjusted\sales_adjusted\sales_2024-03.csv
   missing_value_adjusted\sales_adjusted\sales_2024-04.csv
   missing_value_adjusted\sales_adjusted\sales_2024-05.csv
  ... and 27 more

duplicate_row
Folder: c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\duplicate_row_adjusted
CSV files found: 32
   duplicate_row_adjusted\sales_adjusted\sales_2024-01.csv
   duplicate_row_adjusted\sales_adjusted\sales_2024-02.csv
   duplicate_row_adjusted\sales_adjusted\sales_2024-03.csv
   duplicate_row_adjusted\sales_adjusted\sales_2024-04.csv
   duplicate_row_adjusted\sales_adjusted\sales_2024-05.csv
  ... and 27 more

invalid_data
Folder: c:\Users\HP\OneDrive\Desktop\retail_sales_analytic

In [9]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
SYNTHETIC_DIR = PROJECT_ROOT / "data" / "synthetic"
CONTROL_DIR = SYNTHETIC_DIR / "control_outputs"

ANOMALY_TRUTH_FILE = CONTROL_DIR / "anomaly_truth.csv"

ANOMALY_SCENARIOS = {
    "missing_value": SYNTHETIC_DIR / "missing_value_adjusted",
    "duplicate_row": SYNTHETIC_DIR / "duplicate_row_adjusted",
    "invalid_data": SYNTHETIC_DIR / "invalid_data_adjusted",
    "outlier": SYNTHETIC_DIR / "outlier_adjusted",
}

records = []

print("=" * 70)
print("02.33 - REBUILD ANOMALY TRUTH")
print("=" * 70)

for anomaly_type, scenario_dir in ANOMALY_SCENARIOS.items():

    files = sorted(scenario_dir.rglob("*.csv"))

    print(f"\n{anomaly_type}: {len(files)} files")

    for file_path in files:

        records.append({
            "business_key": file_path.stem,
            "anomaly_type": anomaly_type,
            "expected_classification": "REVIEW",
            "source_path": str(
                file_path.relative_to(SYNTHETIC_DIR)
            )
        })

anomaly_truth = pd.DataFrame(
    records,
    columns=[
        "business_key",
        "anomaly_type",
        "expected_classification",
        "source_path"
    ]
)

# Save
CONTROL_DIR.mkdir(parents=True, exist_ok=True)

anomaly_truth.to_csv(
    ANOMALY_TRUTH_FILE,
    index=False
)

print("\n" + "=" * 70)
print("ANOMALY TRUTH CREATED")
print("=" * 70)

print(f"Total records : {len(anomaly_truth):,}")
print(f"Columns       : {len(anomaly_truth.columns)}")

print("\nAnomaly distribution:")
print(anomaly_truth["anomaly_type"].value_counts())

print("\nSaved to:")
print(ANOMALY_TRUTH_FILE)

print("\n02.33 anomaly_truth rebuild completed.")


02.33 - REBUILD ANOMALY TRUTH

missing_value: 32 files

duplicate_row: 32 files

invalid_data: 32 files

outlier: 32 files

ANOMALY TRUTH CREATED
Total records : 128
Columns       : 4

Anomaly distribution:
anomaly_type
missing_value    32
duplicate_row    32
invalid_data     32
outlier          32
Name: count, dtype: int64

Saved to:
c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\control_outputs\anomaly_truth.csv

02.33 anomaly_truth rebuild completed.


In [10]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
CONTROL_DIR = PROJECT_ROOT / "data" / "synthetic" / "control_outputs"

print("=" * 70)
print("02.33 - STEP 4 - VALIDATE SIMULATION CONTROL OUTPUTS")
print("=" * 70)

all_valid = True

# ---------------------------------------------------------
# 1. simulation_manifest.json
# ---------------------------------------------------------
manifest_file = CONTROL_DIR / "simulation_manifest.json"

print("\nFile: simulation_manifest.json")

if manifest_file.exists():
    try:
        with open(manifest_file, "r", encoding="utf-8") as f:
            manifest = json.load(f)

        print("Status : EXISTS")
        print("Type   : JSON")
        print("Keys   :", len(manifest))
        print("Valid  : YES")

    except Exception as e:
        print("Status : EXISTS")
        print("Type   : JSON")
        print("Valid  : NO")
        print("Error  :", e)
        all_valid = False

else:
    print("Status : MISSING")
    all_valid = False


# ---------------------------------------------------------
# 2. anomaly_truth.csv
# ---------------------------------------------------------
anomaly_file = CONTROL_DIR / "anomaly_truth.csv"

print("\nFile: anomaly_truth.csv")

if anomaly_file.exists():
    try:
        anomaly_df = pd.read_csv(anomaly_file)

        required_anomaly_columns = [
            "business_key",
            "anomaly_type",
            "expected_classification",
            "source_path"
        ]

        missing_columns = [
            col for col in required_anomaly_columns
            if col not in anomaly_df.columns
        ]

        print("Status :", "EXISTS")
        print("Type   : CSV")
        print("Rows   :", len(anomaly_df))
        print("Columns:", len(anomaly_df.columns))

        if missing_columns:
            print("Missing columns:", missing_columns)
            print("Valid  : NO")
            all_valid = False

        elif len(anomaly_df) != 128:
            print("Expected rows: 128")
            print("Valid  : NO")
            all_valid = False

        elif anomaly_df["anomaly_type"].isna().any():
            print("Valid  : NO")
            print("Reason : Missing anomaly type")
            all_valid = False

        else:
            print("Valid  : YES")

    except Exception as e:
        print("Status : EXISTS")
        print("Valid  : NO")
        print("Error  :", e)
        all_valid = False

else:
    print("Status : MISSING")
    all_valid = False


# ---------------------------------------------------------
# 3. expected_control_totals.csv
# ---------------------------------------------------------
totals_file = CONTROL_DIR / "expected_control_totals.csv"

print("\nFile: expected_control_totals.csv")

if totals_file.exists():
    try:
        totals_df = pd.read_csv(totals_file)

        print("Status :", "EXISTS")
        print("Type   : CSV")
        print("Rows   :", len(totals_df))
        print("Columns:", len(totals_df.columns))
        print("Valid  : YES")

    except Exception as e:
        print("Status : EXISTS")
        print("Valid  : NO")
        print("Error  :", e)
        all_valid = False

else:
    print("Status : MISSING")
    all_valid = False


# ---------------------------------------------------------
# 4. generation_run_log.txt
# ---------------------------------------------------------
log_file = CONTROL_DIR / "generation_run_log.txt"

print("\nFile: generation_run_log.txt")

if log_file.exists():
    try:
        log_text = log_file.read_text(encoding="utf-8")

        print("Status : EXISTS")
        print("Type   : TXT")
        print("Size   :", len(log_text), "characters")
        print("Valid  : YES")

    except Exception as e:
        print("Status : EXISTS")
        print("Valid  : NO")
        print("Error  :", e)
        all_valid = False

else:
    print("Status : MISSING")
    all_valid = False


# ---------------------------------------------------------
# FINAL RESULT
# ---------------------------------------------------------
print("\n" + "=" * 70)

if all_valid:
    print("02.33 Step 4 validation: PASSED")
    print("All simulation control outputs are valid.")
else:
    print("02.33 Step 4 validation: NEEDS REVIEW")

print("=" * 70)

02.33 - STEP 4 - VALIDATE SIMULATION CONTROL OUTPUTS

File: simulation_manifest.json
Status : EXISTS
Type   : JSON
Keys   : 7
Valid  : YES

File: anomaly_truth.csv
Status : EXISTS
Type   : CSV
Rows   : 128
Columns: 4
Valid  : YES

File: expected_control_totals.csv
Status : EXISTS
Type   : CSV
Rows   : 34
Columns: 4
Valid  : YES

File: generation_run_log.txt
Status : EXISTS
Type   : TXT
Size   : 242 characters
Valid  : YES

02.33 Step 4 validation: PASSED
All simulation control outputs are valid.


In [11]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
CONTROL_DIR = PROJECT_ROOT / "data" / "synthetic" / "control_outputs"

MANIFEST_FILE = CONTROL_DIR / "simulation_manifest.json"
ANOMALY_FILE = CONTROL_DIR / "anomaly_truth.csv"
TOTALS_FILE = CONTROL_DIR / "expected_control_totals.csv"
LOG_FILE = CONTROL_DIR / "generation_run_log.txt"

print("=" * 70)
print("02.33 - STEP 5 - FINAL SIMULATION CONTROL OUTPUT VALIDATION")
print("=" * 70)

checks = []

# ---------------------------------------------------------
# 1. Manifest
# ---------------------------------------------------------
manifest_valid = False

if MANIFEST_FILE.exists():
    try:
        with open(MANIFEST_FILE, "r", encoding="utf-8") as f:
            manifest = json.load(f)

        manifest_valid = isinstance(manifest, dict) and len(manifest) > 0
    except Exception:
        manifest_valid = False

checks.append(("simulation_manifest.json", manifest_valid))

# ---------------------------------------------------------
# 2. Anomaly truth
# ---------------------------------------------------------
anomaly_valid = False

if ANOMALY_FILE.exists():
    try:
        anomaly_df = pd.read_csv(ANOMALY_FILE)

        required_columns = [
            "business_key",
            "anomaly_type",
            "expected_classification",
            "source_path"
        ]

        anomaly_valid = (
            len(anomaly_df) == 128
            and all(col in anomaly_df.columns for col in required_columns)
            and anomaly_df["anomaly_type"].notna().all()
            and anomaly_df["source_path"].notna().all()
        )

    except Exception:
        anomaly_valid = False

checks.append(("anomaly_truth.csv", anomaly_valid))

# ---------------------------------------------------------
# 3. Expected control totals
# ---------------------------------------------------------
totals_valid = False

if TOTALS_FILE.exists():
    try:
        totals_df = pd.read_csv(TOTALS_FILE)

        totals_valid = (
            len(totals_df) == 34
            and len(totals_df.columns) == 4
        )

    except Exception:
        totals_valid = False

checks.append(("expected_control_totals.csv", totals_valid))

# ---------------------------------------------------------
# 4. Generation log
# ---------------------------------------------------------
log_valid = False

if LOG_FILE.exists():
    try:
        log_text = LOG_FILE.read_text(encoding="utf-8").strip()
        log_valid = len(log_text) > 0
    except Exception:
        log_valid = False

checks.append(("generation_run_log.txt", log_valid))

# ---------------------------------------------------------
# DISPLAY RESULTS
# ---------------------------------------------------------
print("\nFINAL CONTROL OUTPUT CHECKS")
print("-" * 70)

for file_name, result in checks:
    print(f"{file_name:<35} : {'PASSED' if result else 'FAILED'}")

all_valid = all(result for _, result in checks)

print("\n" + "=" * 70)

if all_valid:
    print("02.33 Step 5 validation: PASSED")
    print("Simulation control outputs are complete and valid.")
else:
    print("02.33 Step 5 validation: NEEDS REVIEW")

print("=" * 70)

02.33 - STEP 5 - FINAL SIMULATION CONTROL OUTPUT VALIDATION

FINAL CONTROL OUTPUT CHECKS
----------------------------------------------------------------------
simulation_manifest.json            : PASSED
anomaly_truth.csv                   : PASSED
expected_control_totals.csv         : PASSED
generation_run_log.txt              : PASSED

02.33 Step 5 validation: PASSED
Simulation control outputs are complete and valid.
